<a href="https://colab.research.google.com/github/Th1rex/Ciencia_Dados_Aprendizagem_Maquina/blob/main/Projeto_DataScience/Limpeza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Etapa 1 — Limpeza e padronização da base real do formulário TechMotors.

Entrada : TechMotors.csv (exportado do Google Forms)
Saída   : base_real_limpa.csv

O que este script faz:
  1. Renomeia as colunas para nomes curtos (sem espaços e acentos)
  2. Converte o carimbo de data/hora para datetime
  3. Converte as escalas 1–5 para inteiro
  4. Separa a pergunta de múltipla escolha ("marque até 2") em colunas binárias
  5. Marca as respostas que ultrapassaram o limite de 2 opções
  6. Sinaliza que a faixa etária só existe a partir de 10/11/2026

In [ ]:
import sys
import pandas as pd

ARQ_ENTRADA = sys.argv[1]
ARQ_SAIDA = sys.argv[2]

1. Leitura e renomeação das colunas

In [ ]:
df = pd.read_csv(ARQ_ENTRADA)

In [ ]:
# O Forms exporta os títulos com espaços extras; removemos antes de mapear
df.columns = [c.strip() for c in df.columns]

NOMES = {
    "Carimbo de data/hora": "data_hora",
    "Qual sua faixa etária?": "faixa_etaria",
    "Qual fator mais pesa na hora de escolher uma oficina?": "fator_escolha",
    "Até que distância de casa ou do trabalho você aceita ir para levar o seu veiculo?": "distancia_aceita",
    "Você trocaria de oficina para pagar menos? A partir de quanto de economia?": "troca_por_economia",
    "Numa escala de 1 a 5, quanto você confia na qualidade do serviço da oficina que usa hoje?": "confianca_oficina_atual",
    "Como você escolheu a oficina que usa hoje?": "como_escolheu",
    "Você costuma ler avaliações (Google, redes sociais) antes de escolher uma oficina?": "le_avaliacoes",
    "O que faria você deixar de ir a uma oficina? (marque até 2)": "motivos_abandono",
    "Quanto é importante saber o preço antes de levar o seu veiculo?": "importancia_preco",
    "Você prefere agendar o serviço ou chegar sem marcar?": "preferencia_agendamento",
    "Pensando na última vez que foi a uma oficina, o que você mais gostou e o que mais incomodou?": "comentario",
}
df = df.rename(columns=NOMES)

In [ ]:
# Remove espaços sobrando em todos os campos de texto
for col in df.columns:
    if pd.api.types.is_string_dtype(df[col]):
        df[col] = df[col].str.strip()

In [ ]:
# ---------------------------------------------------------------
# 2. Data/hora — formato do Forms: "2026/11/04 8:00:44 PM GMT-3"
# ---------------------------------------------------------------
df["data_hora"] = pd.to_datetime(
    df["data_hora"].str.replace(" GMT-3", "", regex=False),
    format="%Y/%m/%d %I:%M:%S %p",
)

In [ ]:
# ---------------------------------------------------------------
# 3. Escalas 1–5 viram inteiro
# ---------------------------------------------------------------
for col in ["confianca_oficina_atual", "importancia_preco"]:
    df[col] = df[col].astype(int)

In [ ]:
# ---------------------------------------------------------------
# 4. Múltipla escolha → colunas binárias (1 = marcou, 0 = não marcou)
# ---------------------------------------------------------------
MOTIVOS = {
    "Serviço mal feito": "motivo_servico_mal_feito",
    "Preço acima do combinado": "motivo_preco_acima",
    "Atraso na entrega": "motivo_atraso",
    "Mau atendimento": "motivo_mau_atendimento",
    "Falta de transparência (peças, orçamento)": "motivo_falta_transparencia",
    "Mudança de endereço": "motivo_mudanca_endereco",
}

listas = df["motivos_abandono"].fillna("").str.split(";")
for texto, coluna in MOTIVOS.items():
    df[coluna] = listas.apply(lambda itens: int(texto in itens))

In [ ]:
# Confere se apareceu alguma opção fora do mapeamento (ex.: "Outro")
opcoes_vistas = {item for itens in listas for item in itens if item}
desconhecidas = opcoes_vistas - set(MOTIVOS)
if desconhecidas:
    print("ATENÇÃO — opções não mapeadas:", desconhecidas)

In [ ]:
# ---------------------------------------------------------------
# 5. Controle do limite "marque até 2"
# ---------------------------------------------------------------
df["qtd_motivos"] = df[list(MOTIVOS.values())].sum(axis=1)
df["acima_limite_2"] = (df["qtd_motivos"] > 2).astype(int)

In [ ]:
# ---------------------------------------------------------------
# 6. Faixa etária — pergunta adicionada ao formulário em 10/11/2026
#    Respostas anteriores ficam como "Não perguntado" (não é recusa)
# ---------------------------------------------------------------
INICIO_PERGUNTA_IDADE = pd.Timestamp("2026-11-10")
df["faixa_etaria"] = df["faixa_etaria"].where(
    df["faixa_etaria"].notna(),
    "Não perguntado",
)
df["idade_perguntada"] = (df["data_hora"] >= INICIO_PERGUNTA_IDADE).astype(int)

In [ ]:
# ---------------------------------------------------------------
# 7. Identificador e origem
# ---------------------------------------------------------------
df.insert(0, "id", [f"R{i:03d}" for i in range(1, len(df) + 1)])
df["origem"] = "real"

df.to_csv(ARQ_SAIDA, index=False, encoding="utf-8-sig")

In [ ]:
# ---------------------------------------------------------------
# Resumo no terminal
# ---------------------------------------------------------------
print(f"Registros: {len(df)} | Colunas: {df.shape[1]}")
print(f"Faixa etária perguntada: {df['idade_perguntada'].sum()} de {len(df)}")
print(f"Acima do limite de 2 motivos: {df['acima_limite_2'].sum()} de {len(df)}")
print("\nDistribuição da quantidade de motivos marcados:")
print(df["qtd_motivos"].value_counts().sort_index().to_string())
print("\nFrequência de cada motivo:")
print(df[list(MOTIVOS.values())].sum().sort_values(ascending=False).to_string())